# Fine-Tuning Indonesian Sarcasm Detection Model using IndoBERT

Notebook ini dibuat otomatis untuk melatih model deteksi sarkasme (2 kelas: Not Sarcasm, Sarcasm) bahasa Indonesia menggunakan arsitektur IndoBERT. Notebook ini dirancang agar siap dijalankan di Google Colab menggunakan akselerasi GPU.

### Alur Langkah:
1. Install dependencies otomatis
2. Hubungkan ke Google Drive
3. Unggah dan load dataset dari CSV/XLSX
4. Preprocessing data & standardisasi label
5. Fine-tuning menggunakan HuggingFace Trainer API
6. Evaluasi model (Accuracy, Precision, Recall, F1, Confusion Matrix)
7. Simpan model hasil training ke Google Drive & ekspor zip

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn openpyxl matplotlib seaborn

## 2. Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Unggah / Buat Dataset

Pastikan dataset Anda memiliki dua kolom: `text` dan `label` (0 = Not Sarcasm, 1 = Sarcasm). Jika file Anda memiliki nama kolom yang berbeda, silakan sesuaikan di bawah.

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import files

print("Unggah file dataset CSV atau XLSX Anda:")
uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]

if dataset_filename.endswith('.csv'):
    df = pd.read_csv(dataset_filename)
else:
    df = pd.read_excel(dataset_filename)

print(f"\nDataset berhasil dimuat: {len(df)} baris.")
print(df.head())

## 4. Preprocessing Data & Label Mapping

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags symbol
    text = text.replace('#', '')
    # Remove emojis & non-ascii
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

# Sesuaikan nama kolom jika berbeda
text_col = 'text'
label_col = 'label'

df['clean_text'] = df[text_col].apply(clean_text)

# Mapping label ke integer
label_map = {
    '0': 0, 'non-sarcasm': 0, 'non sarcasm': 0, 'not sarcasm': 0, 'not-sarcasm': 0, 0: 0, 'bukan sarkas': 0, 'bukan sarkasme': 0,
    '1': 1, 'sarcasm': 1, 'sarcastic': 1, 'sarkas': 1, 'sarkasme': 1, 1: 1
}
df['label_id'] = df[label_col].map(label_map)
df = df.dropna(subset=['label_id'])
df['label_id'] = df['label_id'].astype(int)

print("Distribusi Label:")
print(df['label_id'].value_counts())

## 5. Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42,
    stratify=df['label_id']
)

print(f"Jumlah data training: {len(train_df)}")
print(f"Jumlah data validation: {len(val_df)}")

## 6. Tokenisasi Dataset

In [ ]:
from transformers import AutoTokenizer
import torch
from datasets import Dataset

model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch['clean_text'], truncation=True, padding='max_length', max_length=128)

train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label_id']].rename(columns={'label_id': 'label'}))
val_dataset = Dataset.from_pandas(val_df[['clean_text', 'label_id']].rename(columns={'label_id': 'label'}))

train_tokenized = train_dataset.map(tokenize_fn, batched=True)
val_tokenized = val_dataset.map(tokenize_fn, batched=True)

## 7. Fine-Tuning Model

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_tuple

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_tuple(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Load Model dengan 2 Label kelas
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Mulai Training
trainer.train()

## 8. Evaluasi & Visualisasi

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

sarcasm_labels = ["Not Sarcasm", "Sarcasm"]

# Ambil prediksi
preds_output = trainer.predict(val_tokenized)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

# Print metrik
print(classification_report(y_true, y_pred, target_names=sarcasm_labels))

# Render Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=sarcasm_labels,
            yticklabels=sarcasm_labels)
plt.xlabel('Prediksi')
plt.ylabel('Sebenarnya')
plt.title('Confusion Matrix - Sarcasm')
plt.show()

## 9. Simpan Model & Export ke Google Drive

In [ ]:
import json

# Simpan model lokal
model_save_path = './sarcasm_model'
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

# Simpan label mapping
label_mapping = {
    "label2id": {"Not Sarcasm": 0, "Sarcasm": 1},
    "id2label": {"0": "Not Sarcasm", "1": "Sarcasm"},
    "labels": ["Not Sarcasm", "Sarcasm"]
}
with open(os.path.join(model_save_path, 'label_mapping.json'), 'w') as f:
    json.dump(label_mapping, f, indent=2)

# Zip model
!zip -r sarcasm_model.zip ./sarcasm_model

# Copy ke Google Drive
gdrive_dest = '/content/drive/MyDrive/Sarcasm_Model/'
os.makedirs(gdrive_dest, exist_ok=True)
!cp sarcasm_model.zip {gdrive_dest}
print(f"\nModel sukses disimpan ke Google Drive Anda di: {gdrive_dest}")